![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 0 — Arquitectura, Data Fabric y Productos
**Rol:** CRB_ARQUITECTURA | **Tiempo:** 15 min | **Criterio:** Navegar la arquitectura, productos de datos, contratos e interoperabilidad

In [ ]:
USE ROLE CRB_ARQUITECTURA;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: La arquitectura ya existe

Verificamos la arquitectura de dominios. Los 13 schemas corresponden a áreas de negocio de CredibanCo — cada dominio con aislamiento y permisos propios.

In [ ]:
-- 13 dominios organizados como schemas
SHOW SCHEMAS IN DATABASE CREDIBANCO_HOL;

Los roles personalizados controlan quién accede a qué. Cada área de CredibanCo tiene un rol exclusivo con permisos granulares.

In [ ]:
-- Aislamiento por dominio con database roles
SHOW ROLES LIKE 'CRB%';

Los tags de gobierno permiten clasificar y auditar datos sensibles. Snowflake aplica gobierno **sin impactar performance**.

In [ ]:
-- Tags de gobierno aplicados a objetos
SHOW TAGS IN SCHEMA CREDIBANCO_HOL.GOBIERNO;

Exploramos los objetos disponibles en el dominio Arquitectura y la integración con GitHub para versionamiento de código.

In [ ]:
-- Objetos disponibles en el dominio Arquitectura
SHOW OBJECTS IN SCHEMA CREDIBANCO_HOL.ARQUITECTURA;

-- Interoperabilidad: integración Git configurada
SHOW GIT REPOSITORIES IN SCHEMA CREDIBANCO_HOL.PLATAFORMA;

## Bloque 2 — Ejecutar: Producto de datos + linaje

In [ ]:
-- Crear un producto de datos gobernado
CREATE OR REPLACE VIEW CREDIBANCO_HOL.ARQUITECTURA.PRODUCTO_PAGOS_V1 AS
SELECT a.AUTORIZACION_ID, a.CIUDAD, a.MCC, a.MONTO, a.CODIGO_RESPUESTA,
       c.RAZON_SOCIAL AS COMERCIO
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES a
JOIN CREDIBANCO_HOL.COMERCIOS.COMERCIOS c ON a.COMERCIO_ID = c.COMERCIO_ID;

In [ ]:
-- Aplicar tag de dominio al producto
ALTER VIEW CREDIBANCO_HOL.ARQUITECTURA.PRODUCTO_PAGOS_V1
  SET TAG CREDIBANCO_HOL.GOBIERNO.TAG_DOMINIO = 'PAGOS';

El linaje nativo muestra la trazabilidad completa: de dónde vienen los datos y quién los consume. Clave para auditoría y compliance.

In [ ]:
-- Linaje programático: ¿de dónde viene este producto?
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.ARQUITECTURA.PRODUCTO_PAGOS_V1', 'view', 'upstream', 3
));

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Muéstrame el linaje completo de la tabla AUTORIZACIONES: qué objetos dependen de ella, qué políticas la protegen y qué tags tiene aplicados. Genera un resumen ejecutivo.**

In [ ]:
-- Verificación final
SELECT 'T0_COMPLETO' AS status, COUNT(*) AS filas_producto
FROM CREDIBANCO_HOL.ARQUITECTURA.PRODUCTO_PAGOS_V1;